In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
import networkx as nx
import torch
import torch.nn.functional as F


PROJECT_DIR = Path().resolve()
DATA_DIR = PROJECT_DIR / "data"
SCRIPT_DIR = PROJECT_DIR

In [2]:
import utils

data = utils.read_data(DATA_DIR, DATA_DIR)

Found 0 zip file(s): []
DE: Symmetric = False
MD: Symmetric = False


In [3]:
data["DE"].keys()

dict_keys(['filepath', 'adj', 'edges', 'edge_features_traffic', 'accidents'])

In [4]:
WEATHER_COLS = ["tavg", "tmin", "tmax", "prcp", "wspd"] 
def build_node_cache(state_dir: Path):
    files = list((state_dir / "Nodes").glob("node_features_*.csv"))
    months = sorted(tuple(int(p) for p in f.stem.split("_")[-2:]) for f in files)

    first = pd.read_csv(state_dir / "Nodes" / f"node_features_{months[0][0]}_{months[0][1]}.csv")
    N = len(first)
    coords = first[["lon", "lat"]].to_numpy(np.float32)            # [N, 2] = (lat, lon)

    weather = np.empty((len(months), N, len(WEATHER_COLS)), dtype=np.float32)
    for t, (y, m) in enumerate(months):
        df = pd.read_csv(state_dir / "Nodes" / f"node_features_{y}_{m}.csv",
                         usecols=WEATHER_COLS, dtype=np.float32)
        weather[t] = df[WEATHER_COLS].to_numpy()                     # NaNs kept as NaN

    np.savez(state_dir / "node_cache.npz",
             weather=weather,
             months=np.array(months),
             coords=coords) 
build_node_cache(DATA_DIR / "DE")
build_node_cache(DATA_DIR / "MD")


In [ ]:
def degree_onehot(edge_index, N, max_deg=None):
    out_deg = torch.bincount(edge_index[0], minlength=N)
    in_deg = torch.bincount(edge_index[1], minlength=N)
    if max_deg is None:
        max_deg = int(max(out_deg.max(), in_deg.max()))
    eye = torch.eye(max_deg + 1)
    return torch.cat([eye[in_deg.clamp(max=max_deg)],
                      eye[out_deg.clamp(max=max_deg)]], dim=1)


def betweenness(edge_index, N, k=500, seed=0):
    G = nx.DiGraph()
    G.add_nodes_from(range(N))
    G.add_edges_from(edge_index.t().tolist())
    bc = nx.betweenness_centrality(G, k=min(k, N), normalized=True, seed=seed)
    return torch.tensor([bc[i] for i in range(N)], dtype=torch.float32).unsqueeze(1)


def node2vec(edge_index, N, dim=128, walk_length=20, walks_per_node=10, window=5,
             epochs=1, batch_size=1024, lr=0.01, device="cuda", seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    ei = torch.cat([edge_index, edge_index.flip(0)], dim=1).to(device)
    order = torch.argsort(ei[0] * N + ei[1])
    src, col = ei[0][order], ei[1][order]
    deg = torch.bincount(src, minlength=N)
    rowptr = torch.zeros(N + 1, dtype=torch.long, device=device)
    rowptr[1:] = torch.cumsum(deg, 0)

    def walks(start):
        path, cur = [start], start
        for _ in range(walk_length - 1):
            d = deg[cur]
            off = (torch.rand(cur.shape, generator=g, device=device) * d.clamp(min=1)).long()
            nxt = col[(rowptr[cur] + off).clamp(max=col.numel() - 1)]
            cur = torch.where(d > 0, nxt, cur)
            path.append(cur)
        return torch.stack(path, dim=1)

    emb = torch.nn.Embedding(N, dim).to(device)
    ctx = torch.nn.Embedding(N, dim).to(device)
    torch.nn.init.normal_(emb.weight, std=0.1)
    torch.nn.init.zeros_(ctx.weight)
    opt = torch.optim.Adam(list(emb.parameters()) + list(ctx.parameters()), lr=lr)

    for ep in range(epochs):
        starts = torch.arange(N, device=device).repeat(walks_per_node)
        starts = starts[torch.randperm(starts.numel(), generator=g, device=device)]
        for b in range(0, starts.numel(), batch_size):
            w = walks(starts[b:b + batch_size])
            u = torch.cat([w[:, :-o] for o in range(1, window + 1)], 1).reshape(-1)
            v = torch.cat([w[:, o:] for o in range(1, window + 1)], 1).reshape(-1)
            neg = torch.randint(0, N, (u.numel(),), generator=g, device=device)
            zu = emb(u)
            loss = -(F.logsigmoid((zu * ctx(v)).sum(-1))
                     + F.logsigmoid(-(zu * ctx(neg)).sum(-1))).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"  node2vec epoch {ep}: loss {loss.item():.4f}")
    return emb.weight.detach().cpu()                                 # [N, dim]


def build_static_cache(state_dir: Path, k_bc=500, device="cuda"):
    out = state_dir / "static_node_cache.npz"
    if out.exists():
        return
    adj = torch.load(state_dir / "adj_matrix.pt", map_location="cpu", weights_only=False).coalesce()
    edge_index, N = adj.indices(), adj.shape[0]
    t = time.time(); deg = degree_onehot(edge_index, N);  print(f"degree {tuple(deg.shape)} {time.time()-t:.0f}s")
    t = time.time(); bc = betweenness(edge_index, N, k_bc); print(f"centrality {tuple(bc.shape)} {time.time()-t:.0f}s")
    t = time.time(); n2v = node2vec(edge_index, N, device=device); print(f"node2vec {tuple(n2v.shape)} {time.time()-t:.0f}s")
    np.savez(out, degree=deg.numpy(), centrality=bc.numpy(), node2vec=n2v.numpy())


build_static_cache(DATA_DIR / "DE")
build_static_cache(DATA_DIR / "MD")

degree (49023, 14) 0s
centrality (49023, 1) 67s
  node2vec epoch 0: loss 0.0175
node2vec (49023, 128) 4s
degree (250565, 18) 0s
centrality (250565, 1) 310s
  node2vec epoch 0: loss 0.0052
node2vec (250565, 128) 27s


In [5]:
def build_edge_index(adj):
    adj = adj.coalesce()
    edge_index = adj.indices()                                   # [2, E], sorted by (u, v)
    edge_key = edge_index[0] * adj.shape[0] + edge_index[1]      # [E], ascending -> used for lookups
    return edge_index, edge_key


def to_edge_order(sp, edge_key, N):
    # Map a raw (uncoalesced) sparse [N, N] tensor onto the canonical edge order.
    # Duplicates are averaged ignoring NaN; edges with no value -> NaN.
    idx, val = sp._indices(), sp._values().float()
    k = torch.searchsorted(edge_key, idx[0] * N + idx[1])
    assert torch.equal(edge_key[k], idx[0] * N + idx[1]), "entry not in adj"
    ok = ~torch.isnan(val)
    s = torch.zeros(edge_key.numel()).index_add_(0, k[ok], val[ok])
    c = torch.zeros(edge_key.numel()).index_add_(0, k[ok], torch.ones(int(ok.sum())))
    out = torch.full((edge_key.numel(),), float("nan"))
    out[c > 0] = s[c > 0] / c[c > 0]
    return out                                                   # [E]


def build_static_edge_attr(edges, edge_key, N, categories=None):
    # categories: fixed list of names (e.g. union over DE + MD); categories a state lacks -> 0
    if categories is None:
        categories = sorted(str(k) for k in edges if k not in ("oneway", "length"))
    keys = {str(k): k for k in edges}                            # np.str_ -> str
    cols = [torch.log1p(to_edge_order(edges["length"], edge_key, N)),
            to_edge_order(edges["oneway"], edge_key, N)]
    names = ["log_length", "oneway"]
    for c in categories:
        cols.append(to_edge_order(edges[keys[c]], edge_key, N) if c in keys
                    else torch.zeros(edge_key.numel()))
        names.append(f"cat_{c}")
    attr = torch.stack(cols, dim=1)
    assert not torch.isnan(attr).any()
    return attr, names                                           # [E, 2 + n_categories]


def build_aadt(traffic, edge_key, N):
    # {year: {"AADT": sparse}} -> {year: [E] log1p(AADT), NaN where not reported}
    return {year: torch.log1p(to_edge_order(d["AADT"], edge_key, N)) for year, d in traffic.items()}

In [6]:
from torch.utils.data import Dataset
from torch_geometric.data import Data


class TrafficDataset(Dataset):
    """One state, a set of years -> one PyG graph per month.

    Each item is Data(x [N, F_node], edge_index [2, E], edge_attr [E, F_edge], y [E], year, month),
    where y is the accident count per edge (use (y > 0).float() for classification).

    Standardization statistics are fitted on this dataset's own months unless `stats` is passed:
    build the train split first, then pass `train_ds.stats` to the val/test splits.

    use_aadt: append [AADT, AADT missing] to edge_attr. Off by default, as in the authors' code
    (--load_dynamic_edge_features).

    data_dict: output of utils.read_data
        data: dict[str, dict]
        ├── "DE"
        │   ├── "filepath":  pathlib.Path            .../data/DE
        │   ├── "adj":       torch sparse COO, float32, shape [N, N], COALESCED
        │   ├── "edges":     dict[str, torch sparse COO [N, N]], NOT coalesced
        │   │     ├── "oneway"        values 0./1.
        │   │     ├── "length"        values in meters
        │   │     └── np.str_(category) values 0./1.  (one key per road category)
        │   ├── "edge_features_traffic": dict[int, dict[str, torch sparse COO [N, N]]], NOT coalesced
        │   │     └── 2009 … 2019 → {"AADT": sparse, values float or NaN}
        │   └── "accidents": pandas.DataFrame
        │         columns: "Unnamed: 0", year, month, node_1, node_2, acc_count, node_1_idx, node_2_idx
        └── "MD"   (same structure)
    """

    def __init__(self, data_dict: dict, state: str, years, stats=None,
                 use_node2vec=True, use_aadt=False, categories=None):
        info = data_dict[state]
        self.state = state
        self.use_node2vec = use_node2vec
        self.use_aadt = use_aadt
        self.N = info["adj"].shape[0]

        # --- edges ---
        self.edge_index, self.edge_key = build_edge_index(info["adj"])
        self.static_edge_attr, self.edge_names = build_static_edge_attr(
            info["edges"], self.edge_key, self.N, categories)
        self.aadt = build_aadt(info["edge_features_traffic"], self.edge_key, self.N) if use_aadt else None

        # --- nodes ---
        node_cache = np.load(info["filepath"] / "node_cache.npz")
        static_cache = np.load(info["filepath"] / "static_node_cache.npz")
        self.coords = torch.from_numpy(node_cache["coords"])                  # [N, 2]
        self.weather = torch.from_numpy(node_cache["weather"])                # [T, N, 5]
        self.month_to_t = {(int(y), int(m)): t for t, (y, m) in enumerate(node_cache["months"])}
        self.degree = torch.from_numpy(static_cache["degree"])                # [N, 2*(max_deg+1)]
        self.centrality = torch.log1p(torch.from_numpy(static_cache["centrality"]) * 1e3)  # [N, 1], skewed -> log
        self.node2vec = torch.from_numpy(static_cache["node2vec"])            # [N, 128]

        # --- labels: edge position of every accident row, grouped by month ---
        acc = info["accidents"]
        key = torch.from_numpy(acc.node_1_idx.to_numpy() * self.N + acc.node_2_idx.to_numpy())
        k = torch.searchsorted(self.edge_key, key)
        assert torch.equal(self.edge_key[k], key), "accident edge not in adj"
        count = torch.from_numpy(acc.acc_count.to_numpy()).float()
        self.labels = {}
        for (y, m), rows in acc.groupby(["year", "month"]).indices.items():
            self.labels[(int(y), int(m))] = (k[rows], count[rows])

        # months in the requested years that have node features (y = 0 if no accident rows)
        years = set(years)
        self.months = sorted(ym for ym in self.month_to_t if ym[0] in years)

        self.stats = stats if stats is not None else self.fit_stats()

    def fit_stats(self):
        ts = [self.month_to_t[ym] for ym in self.months]
        w = self.weather[ts]                                                   # [T_split, N, 5]
        ok = ~torch.isnan(w)
        w_mean = torch.nansum(w, dim=(0, 1)) / ok.sum(dim=(0, 1))
        w_std = torch.sqrt(torch.nansum((w - w_mean) ** 2, dim=(0, 1)) / ok.sum(dim=(0, 1)))
        stats = {
            "coords": (self.coords.mean(0), self.coords.std(0)),
            "centrality": (self.centrality.mean(0), self.centrality.std(0)),
            "weather": (w_mean, w_std),
            "log_length": (self.static_edge_attr[:, 0].mean(), self.static_edge_attr[:, 0].std()),
        }
        if self.use_aadt:
            a = torch.cat([self.aadt_raw(y) for y in sorted({y for y, _ in self.months})])
            a = a[~torch.isnan(a)]
            stats["aadt"] = (a.mean(), a.std())
        return stats

    def aadt_raw(self, year):
        # nearest reported year: carry forward after the last report, backward before the first
        years = sorted(self.aadt)
        return self.aadt[min(max(year, years[0]), years[-1])]                 # [E], log1p, NaN = missing

    @staticmethod
    def standardize(v, mean_std):
        mean, std = mean_std
        return (v - mean) / (std + 1e-8)

    def __len__(self):
        return len(self.months)

    def __getitem__(self, i):
        year, month = self.months[i]
        s = self.stats

        # node features: [coords | degree one-hot | centrality | node2vec | weather | weather missing]
        w = self.weather[self.month_to_t[(year, month)]]                      # [N, 5]
        w_missing = torch.isnan(w).float()
        w = torch.nan_to_num(self.standardize(w, s["weather"]), nan=0.0)
        x = [self.standardize(self.coords, s["coords"]),
             self.degree,
             self.standardize(self.centrality, s["centrality"])]
        if self.use_node2vec:
            x.append(self.node2vec)
        x += [w, w_missing]
        x = torch.cat(x, dim=1)

        # edge features: [log_length | oneway | categories | (aadt | aadt missing)]
        edge_attr = self.static_edge_attr.clone()
        edge_attr[:, 0] = self.standardize(edge_attr[:, 0], s["log_length"])
        if self.use_aadt:
            a = self.aadt_raw(year)
            a_missing = torch.isnan(a).float()
            a = torch.nan_to_num(self.standardize(a, s["aadt"]), nan=0.0)
            edge_attr = torch.cat([edge_attr, a[:, None], a_missing[:, None]], dim=1)

        # labels: accident count per edge, 0 where the CSV has no row
        y = torch.zeros(self.edge_index.shape[1])
        if (year, month) in self.labels:
            k, count = self.labels[(year, month)]
            y[k] = count

        return Data(x=x, edge_index=self.edge_index, edge_attr=edge_attr, y=y,
                    year=year, month=month, num_nodes=self.N)


SPLITS = {
    "DE": {"train": range(2009, 2013), "val": range(2013, 2018), "test": range(2018, 2023)},
    "MD": {"train": range(2015, 2018), "val": range(2018, 2020), "test": range(2020, 2023)},
}

state = "DE"
use_aadt = False
train_ds = TrafficDataset(data, state, SPLITS[state]["train"], use_aadt=use_aadt)
val_ds = TrafficDataset(data, state, SPLITS[state]["val"], stats=train_ds.stats, use_aadt=use_aadt)
test_ds = TrafficDataset(data, state, SPLITS[state]["test"], stats=train_ds.stats, use_aadt=use_aadt)
print(len(train_ds), len(val_ds), len(test_ds))
train_ds[0]

c:\Users\User\miniconda3\envs\graph_ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\User\miniconda3\envs\graph_ml\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\User\AppData\Local\Temp\ipykernel_2076\3837635921.py:62: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  count = torch.fr

48 60 57


Data(x=[49023, 155], edge_index=[2, 116196], edge_attr=[116196, 16], y=[116196], year=2009, month=1, num_nodes=49023)

In [7]:
import torch.nn as nn
import torch_geometric.nn as gnn

class BaselineModel(nn.Module):
    def __init__(self, input_dim, edge_dim, hidden_dim=256, encoder="sage"):
        super().__init__()
        match encoder:
            case "sage":
                enc_layer = gnn.conv.SAGEConv
            case "linear":
                enc_layer = nn.Linear
            case _:
                raise ValueError(f"{encoder} encoder not implemented")

        self.is_gnn = encoder != "linear"
        self.encoder = nn.ModuleList([enc_layer(input_dim, hidden_dim),
                                      enc_layer(hidden_dim, hidden_dim)])
        self.head = nn.Sequential(nn.Linear(2 * hidden_dim + edge_dim, hidden_dim),
                                  nn.ReLU(),
                                  nn.Linear(hidden_dim, 1))

    def encode(self, x, edge_index):
        for i, layer in enumerate(self.encoder):
            x = layer(x, edge_index) if self.is_gnn else layer(x)
            if i < len(self.encoder) - 1:
                x = x.relu()
        return x

    def forward(self, x, edge_index, edge_attr):
        h = self.encode(x, edge_index)
        src, dst = edge_index
        z = torch.cat([h[src], h[dst], edge_attr], dim=1)
        return self.head(z).squeeze(-1)


In [8]:
sample = train_ds[0]
input_dim = sample.x.shape[1]          # features per node (x is [N, F_node])
edge_dim = sample.edge_attr.shape[1]   # features per road (edge_attr is [E, F_edge])
input_dim, edge_dim

(155, 16)

In [ ]:
import time
import torch.nn.functional as F
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 100
MIN_POS = 10


def score_edges(model, h, edge_index, edge_attr, idx):
    src, dst = edge_index[:, idx]
    return model.head(torch.cat([h[src], h[dst], edge_attr[idx]], dim=1)).squeeze(-1)


def auroc(scores, labels):
    # rank-based ROC AUC (equivalent to sklearn.metrics.roc_auc_score, ignoring ties)
    order = torch.argsort(scores)
    ranks = torch.empty_like(order, dtype=torch.float64)
    ranks[order] = torch.arange(1, len(scores) + 1, dtype=torch.float64, device=scores.device)
    n_pos = labels.sum().double()
    n_neg = len(labels) - n_pos
    return ((ranks[labels.bool()].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)).item()


def train_month(model, month, optimizer, task):
    month = month.to(device)
    pos = torch.nonzero(month.y > 0).squeeze(1)
    if pos.numel() < MIN_POS:
        return None
    neg_pool = torch.nonzero(month.y == 0).squeeze(1)
    neg = neg_pool[torch.randperm(neg_pool.numel(), device=device)[:pos.numel()]]   # 1:1, real roads
    idx = torch.cat([pos, neg])

    model.train()
    h = model.encode(month.x, month.edge_index)
    out = score_edges(model, h, month.edge_index, month.edge_attr, idx)              # negatives use their own edge_attr
    if task == "classification":
        loss = F.binary_cross_entropy_with_logits(out, (month.y[idx] > 0).float())
    else:
        loss = F.l1_loss(out, month.y[idx])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, ds, task):
    # score all roads each month; average the metric weighted by the number of accident roads
    model.eval()
    total, weight = 0.0, 0
    for month in ds:
        month = month.to(device)
        n_pos = int((month.y > 0).sum())
        if n_pos < MIN_POS:
            continue
        out = model(month.x, month.edge_index, month.edge_attr)
        if task == "classification":
            metric = auroc(out, (month.y > 0).float())
        else:
            metric = (out - month.y).abs().mean().item()
        total += metric * n_pos
        weight += n_pos
    return total / weight


def build_splits(state, use_node2vec=True, use_aadt=False):
    train_ds = TrafficDataset(data, state, SPLITS[state]["train"],
                              use_node2vec=use_node2vec, use_aadt=use_aadt)
    val_ds = TrafficDataset(data, state, SPLITS[state]["val"], stats=train_ds.stats,
                            use_node2vec=use_node2vec, use_aadt=use_aadt)
    test_ds = TrafficDataset(data, state, SPLITS[state]["test"], stats=train_ds.stats,
                             use_node2vec=use_node2vec, use_aadt=use_aadt)
    return train_ds, val_ds, test_ds


def train(task, encoder, train_ds, val_ds, test_ds, seed=0):
    torch.manual_seed(seed)
    sample = train_ds[0]
    model = BaselineModel(input_dim=sample.x.shape[1],
                          edge_dim=sample.edge_attr.shape[1],
                          hidden_dim=256,
                          encoder=encoder).to(device)
    optimizer = Adam(model.parameters(), lr=1e-3)

    metric_name = "AUROC" if task == "classification" else "MAE"
    best_val, best_state = None, None
    for epoch in range(1, EPOCHS + 1):
        t = time.time()
        losses = [l for l in (train_month(model, train_ds[i], optimizer, task)
                              for i in torch.randperm(len(train_ds)).tolist()) if l is not None]
        if epoch % 5 == 0 or epoch == 1:
            val = evaluate(model, val_ds, task)
            better = best_val is None or (val > best_val if task == "classification" else val < best_val)
            if better:
                best_val = val
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            print(f"epoch {epoch:3d}  loss {np.mean(losses):.4f}  val {metric_name} {val:.4f}  ({time.time() - t:.1f}s)")

    model.load_state_dict(best_state)
    test = evaluate(model, test_ds, task)
    print(f"best val {metric_name} {best_val:.4f}  test {metric_name} {test:.4f}")
    return {"val": best_val, "test": test}

In [ ]:
state = "DE"
splits = {n2v: build_splits(state, use_node2vec=n2v, use_aadt=False) for n2v in (True, False)}
configs = [("sage", True), ("linear", True), ("linear", False)]

results = {}
for task in ["classification", "regression"]:
    for enc, n2v in configs:
        name = f"{task}_{enc}_{'n2v' if n2v else 'no-n2v'}"
        print(f"Training {name}")
        results[name] = train(task, enc, *splits[n2v])

pd.DataFrame(results).T